In [ ]:
import cv2 as cv
import argparse
import numpy as np
from ultralytics import YOLO


# ==========================
# Kalman Filter Tracker
# ==========================
class KalmanTracker:
    def __init__(self, bbox):
        self.kf = cv.KalmanFilter(4, 2)

        # State: [x, y, vx, vy]
        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], np.float32)

        # Measurement: [x, y]
        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1

        x, y, w, h = bbox
        cx, cy = x + w / 2, y + h / 2

        self.kf.statePre = np.array([[cx], [cy], [0], [0]], dtype=np.float32)

        self.w = w
        self.h = h

    def predict(self):
        pred = self.kf.predict()
        cx, cy = float(pred[0]), float(pred[1])

        return (cx - self.w / 2, cy - self.h / 2, self.w, self.h)

    def update(self, bbox):
        x, y, w, h = bbox
        cx, cy = x + w / 2, y + h / 2

        measurement = np.array([[cx], [cy]], dtype=np.float32)
        self.kf.correct(measurement)

        self.w = w
        self.h = h


# ==========================
# Draw Predictions
# ==========================
def drawPred(frame, bboxes, labels, confidences):
    for i, box in enumerate(bboxes):
        label = f"{labels[i]}: {confidences[i]:.2f}"

        x, y, w, h = map(int, box)

        cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        labelSize, baseLine = cv.getTextSize(label, cv.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        top = max(y, labelSize[1])

        cv.rectangle(frame, (x, top - labelSize[1]),
                     (x + labelSize[0], top + baseLine),
                     (255, 255, 255), cv.FILLED)

        cv.putText(frame, label, (x, top),
                   cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)


# ==========================
# YOLO Detection
# ==========================
def detect_objects(frame, model, threshold):
    results = model(frame, imgsz=640)[0]

    bboxes, labels, confidences = [], [], []

    for box in results.boxes:
        conf = float(box.conf[0])

        if conf > threshold:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])

            label = model.names[cls]

            bboxes.append((x1, y1, x2 - x1, y2 - y1))
            labels.append(label)
            confidences.append(conf)

    return bboxes, labels, confidences


# ==========================
# Main Processing
# ==========================
def process(args):

    model = YOLO("best.pt")

    stream = cv.VideoCapture(args.video)

    if not stream.isOpened():
        print("Error: Could not open video.")
        return

    fps_input = stream.get(cv.CAP_PROP_FPS)
    delay = int(1000 / fps_input) if fps_input > 0 else 30

    frame_width = int(stream.get(cv.CAP_PROP_FRAME_WIDTH))
    frame_height = int(stream.get(cv.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv.VideoWriter_fourcc(*'mp4v')
    out = cv.VideoWriter(args.output, fourcc, fps_input,
                         (frame_width, frame_height))

    cv.namedWindow("YOLO + Kalman Tracking", cv.WINDOW_NORMAL)
    cv.resizeWindow("YOLO + Kalman Tracking", 900, 700)

    DETECT_EVERY_N_FRAMES = 10
    SKIP_FRAMES = 2

    frame_count = 0
    trackers = []
    bboxes, labels, confidences = [], [], []

    paused = False
    last_frame = None

    while stream.isOpened():

        # Frame skipping
        if not paused and frame_count % SKIP_FRAMES != 0:
            stream.grab()
            frame_count += 1
            continue

        if not paused:
            grabbed, frame = stream.read()
            if not grabbed:
                break
        else:
            frame = last_frame.copy()

        timer = cv.getTickCount()

        # ==========================
        # Detection or Prediction
        # ==========================
        if frame_count % DETECT_EVERY_N_FRAMES == 0 or len(trackers) == 0:

            bboxes, labels, confidences = detect_objects(frame, model, args.thr)

            trackers = []
            for box in bboxes:
                trackers.append(KalmanTracker(box))

            print("Detecting:", labels)

        else:
            # Predict
            new_bboxes = []
            for tracker in trackers:
                pred_box = tracker.predict()
                new_bboxes.append(pred_box)

            bboxes = new_bboxes

        # ==========================
        # Update Kalman with detections
        # ==========================
        if len(trackers) == len(bboxes):
            for i in range(len(trackers)):
                trackers[i].update(bboxes[i])

        # ==========================
        # FPS
        # ==========================
        fps = cv.getTickFrequency() / (cv.getTickCount() - timer)

        # ==========================
        # Draw
        # ==========================
        if bboxes:
            drawPred(frame, bboxes, labels, confidences)

            cv.putText(frame, f"FPS: {int(fps)}", (20, 40),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        else:
            cv.putText(frame, "No objects", (20, 40),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        last_frame = frame.copy()

        if paused:
            h, w = frame.shape[:2]
            cv.putText(frame, "PAUSED", (w - 180, h - 20),
                       cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        out.write(frame)
        cv.imshow("YOLO + Kalman Tracking", frame)

        key = cv.waitKey(delay if not paused else 0) & 0xFF

        if key == 27:
            break
        elif key == 32:
            paused = not paused
        elif key == ord('r'):
            rewind_frames = int(fps_input * 10)
            current_pos = int(stream.get(cv.CAP_PROP_POS_FRAMES))
            stream.set(cv.CAP_PROP_POS_FRAMES, max(0, current_pos - rewind_frames))

        frame_count += 1

    stream.release()
    out.release()
    cv.destroyAllWindows()


# ==========================
# Main
# ==========================
def main():
    parser = argparse.ArgumentParser(description='YOLO + Kalman Tracking')

    parser.add_argument('--thr', type=float, default=0.4)
    parser.add_argument('--video', type=str, required=True)
    parser.add_argument('--output', type=str, default='output.mp4')

    args = parser.parse_args()

    process(args)


if __name__ == '__main__':
    main()